# 🔬 NAFNet-Tiny: Semiconductor Image Restoration
## Team Semigone — Semicon India Hackathon 2025

**Problem:** AI-Based Restoration of Degraded Images for Semiconductor Inspection  
**Model:** NAFNet-Tiny (0.48M params) | Joint Denoise + 2× Super-Resolution  
**Loss:** Charbonnier + 0.1 × SSIM | Non-GAN, pixel-fidelity only  
**Target:** ~15,000 iterations on T4 GPU (~2.5 hours)


## 0. Setup & GPU Check


In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")


## 1. Mount Google Drive & Clone Repo


In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone the repo (or pull latest if already cloned)
import os
REPO_DIR = '/content/semicon-restore'

if os.path.exists(REPO_DIR):
    print("Repo already exists, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/<YOUR-USERNAME>/semicon-restore.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt -q


## 2. Dataset Setup

Upload your KLA dataset to Google Drive with this structure:
```
MyDrive/kla_data/
├── Train/
│   ├── Ground_Truth/    # Clean HR images (.npy or .png)
│   └── Degraded/        # Noisy LR images (.npy or .png)
└── Test/
    └── Test_NoisyLR/
        ├── In_Distribution/
        └── Out_of_Distribution/
```


In [ ]:
# ── Configure paths ──────────────────────────────────────
# ⚠️ EDIT THESE to match your Google Drive folder structure

DATA_DIR = '/content/drive/MyDrive/kla_data'
SAVE_DIR = '/content/drive/MyDrive/semicon-checkpoints'
CONFIG   = 'configs/default.yaml'

# Create checkpoint directory
os.makedirs(SAVE_DIR, exist_ok=True)

# Verify data exists
for sub in ['Train/Ground_Truth', 'Train/Degraded']:
    p = os.path.join(DATA_DIR, sub)
    if os.path.exists(p):
        files = os.listdir(p)
        print(f"✓ {sub}: {len(files)} files")
        if files:
            print(f"  Example: {files[0]}")
    else:
        print(f"✗ {sub}: NOT FOUND at {p}")
        print(f"  → Check your Drive folder structure!")


## 3. Explore the Dataset


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load a sample pair
gt_dir = os.path.join(DATA_DIR, 'Train/Ground_Truth')
deg_dir = os.path.join(DATA_DIR, 'Train/Degraded')

gt_files = sorted(os.listdir(gt_dir))
deg_files = sorted(os.listdir(deg_dir))

print(f"GT files: {len(gt_files)}, Degraded files: {len(deg_files)}")

# Load first sample
sample_gt_path = os.path.join(gt_dir, gt_files[0])
sample_deg_path = os.path.join(deg_dir, deg_files[0])

if sample_gt_path.endswith('.npy'):
    sample_gt = np.load(sample_gt_path)
    sample_deg = np.load(sample_deg_path)
else:
    import cv2
    sample_gt = cv2.imread(sample_gt_path, cv2.IMREAD_GRAYSCALE)
    sample_deg = cv2.imread(sample_deg_path, cv2.IMREAD_GRAYSCALE)

print(f"\nGround Truth: shape={sample_gt.shape}, dtype={sample_gt.dtype}, "
      f"range=[{sample_gt.min():.2f}, {sample_gt.max():.2f}]")
print(f"Degraded:     shape={sample_deg.shape}, dtype={sample_deg.dtype}, "
      f"range=[{sample_deg.min():.2f}, {sample_deg.max():.2f}]")

scale = sample_gt.shape[0] // sample_deg.shape[0] if sample_gt.ndim >= 2 else 1
print(f"\nDetected scale factor: {scale}x")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(sample_deg, cmap='gray')
axes[0].set_title(f'Degraded ({sample_deg.shape})')
axes[0].axis('off')
axes[1].imshow(sample_gt, cmap='gray')
axes[1].set_title(f'Ground Truth ({sample_gt.shape})')
axes[1].axis('off')
plt.suptitle(f'Sample Pair: {deg_files[0]}', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Intensity distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(sample_deg.ravel(), bins=100, color='#ff4545', alpha=0.7)
axes[0].set_title('Degraded - Intensity Distribution')
axes[0].set_xlabel('Pixel Value')

axes[1].hist(sample_gt.ravel(), bins=100, color='#00e676', alpha=0.7)
axes[1].set_title('Ground Truth - Intensity Distribution')
axes[1].set_xlabel('Pixel Value')

plt.tight_layout()
plt.show()

print(f"Degraded  - Mean: {sample_deg.mean():.2f}, Std: {sample_deg.std():.2f}")
print(f"Ground Truth - Mean: {sample_gt.mean():.2f}, Std: {sample_gt.std():.2f}")


## 4. Train NAFNet-Tiny 🚀

This will run for ~15,000 iterations (~2.5 hours on T4).  
Checkpoints are saved to Google Drive every 1,000 iterations.

**While this runs, work on your slides and README!**


In [ ]:
# ── Launch Training ─────────────────────────────────────
# This is the main training command.

!python train.py \
    --config {CONFIG} \
    --data_dir {DATA_DIR} \
    --save_dir {SAVE_DIR} \
    --max_iters 15000


### ↻ Resume from Checkpoint (if Colab disconnected)

If your session died, just re-run the cells above to mount Drive and cd into the repo, then run:


In [ ]:
# # ── Resume Training (uncomment to use) ──────────────────
# # Find the latest checkpoint
# import glob
# ckpts = sorted(glob.glob(os.path.join(SAVE_DIR, 'checkpoint_*.pt')))
# if ckpts:
#     latest = ckpts[-1]
#     print(f"Resuming from: {latest}")
#     !python train.py \
#         --config {CONFIG} \
#         --data_dir {DATA_DIR} \
#         --save_dir {SAVE_DIR} \
#         --resume {latest} \
#         --max_iters 15000
# else:
#     print("No checkpoints found. Starting fresh training.")


## 5. Evaluate on Test Set


In [ ]:
# Find best model weights
import glob

best_path = os.path.join(SAVE_DIR, 'best_model.pt')
if not os.path.exists(best_path):
    ckpts = sorted(glob.glob(os.path.join(SAVE_DIR, 'checkpoint_*.pt')))
    best_path = ckpts[-1] if ckpts else None
    print(f"Using latest checkpoint: {best_path}")
else:
    print(f"Using best model: {best_path}")

# Copy weights to repo for GitHub
os.makedirs('weights', exist_ok=True)
if best_path:
    import shutil
    shutil.copy2(best_path, 'weights/nafnet_tiny.pt')
    size_mb = os.path.getsize('weights/nafnet_tiny.pt') / 1e6
    print(f"Copied to weights/nafnet_tiny.pt ({size_mb:.1f} MB)")


In [ ]:
# ── Run Evaluation on In-Distribution Test Set ─────────
!python evaluate.py \
    --input_dir {DATA_DIR}/Test/Test_NoisyLR/In_Distribution \
    --output_dir outputs/In_Distribution \
    --weights weights/nafnet_tiny.pt \
    --config {CONFIG}


In [ ]:
# ── Run Evaluation on Out-of-Distribution Test Set ──────
!python evaluate.py \
    --input_dir {DATA_DIR}/Test/Test_NoisyLR/Out_of_Distribution \
    --output_dir outputs/Out_of_Distribution \
    --weights weights/nafnet_tiny.pt \
    --config {CONFIG}


## 6. Evaluate on Validation Split (with metrics)


In [ ]:
# If you have GT for validation, compute PSNR/SSIM
# Split some training data as validation for metric computation

gt_files_all = sorted(os.listdir(os.path.join(DATA_DIR, 'Train/Ground_Truth')))
deg_files_all = sorted(os.listdir(os.path.join(DATA_DIR, 'Train/Degraded')))

n_val = max(1, len(gt_files_all) // 10)  # Last 10%
val_gt_files = gt_files_all[-n_val:]
val_deg_files = deg_files_all[-n_val:]

print(f"Validation split: {n_val} images (last 10% of training set)")

# Create temp validation dirs
import shutil, tempfile
val_dir = tempfile.mkdtemp()
val_deg_dir = os.path.join(val_dir, 'degraded')
val_gt_dir = os.path.join(val_dir, 'gt')
val_out_dir = os.path.join(val_dir, 'output')
os.makedirs(val_deg_dir); os.makedirs(val_gt_dir); os.makedirs(val_out_dir)

for f in val_deg_files:
    shutil.copy2(os.path.join(DATA_DIR, 'Train/Degraded', f), val_deg_dir)
for f in val_gt_files:
    shutil.copy2(os.path.join(DATA_DIR, 'Train/Ground_Truth', f), val_gt_dir)

!python evaluate.py \
    --input_dir {val_deg_dir} \
    --output_dir {val_out_dir} \
    --weights weights/nafnet_tiny.pt \
    --config {CONFIG} \
    --gt_dir {val_gt_dir}

# Cleanup
shutil.rmtree(val_dir)


## 7. Visualize Results


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Load some test results
in_dist_dir = 'outputs/In_Distribution'
test_dir = os.path.join(DATA_DIR, 'Test/Test_NoisyLR/In_Distribution')

if os.path.exists(in_dist_dir):
    restored_files = sorted(os.listdir(in_dist_dir))[:6]
    test_files = sorted(os.listdir(test_dir))[:6]

    n = min(len(restored_files), 6)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for i in range(n):
        # Load degraded input
        tp = os.path.join(test_dir, test_files[i])
        if tp.endswith('.npy'):
            degraded = np.load(tp)
        else:
            degraded = cv2.imread(tp, cv2.IMREAD_GRAYSCALE)

        # Load restored output
        rp = os.path.join(in_dist_dir, restored_files[i])
        if rp.endswith('.npy'):
            restored = np.load(rp)
        else:
            restored = cv2.imread(rp, cv2.IMREAD_GRAYSCALE)

        axes[0, i].imshow(degraded, cmap='gray')
        axes[0, i].set_title(f'Input', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(restored, cmap='gray')
        axes[1, i].set_title(f'Restored', fontsize=10)
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Degraded', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('NAFNet Output', fontsize=12, fontweight='bold')
    plt.suptitle('Before → After Restoration (In-Distribution)', fontsize=14)
    plt.tight_layout()
    plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to results_comparison.png")
else:
    print(f"No outputs found at {in_dist_dir}. Run evaluation first.")


## 8. Model Info & Inference Speed


In [ ]:
import torch
import time
import yaml

# Load model
with open(CONFIG) as f:
    config = yaml.safe_load(f)

from models.nafnet_sr import build_model
model = build_model(config)
model.eval()

# Parameter count
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: NAFNet-Tiny")
print(f"Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"Model size: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6:.1f} MB")

# Inference speed benchmark
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

dummy = torch.randn(1, 1, 128, 128).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Benchmark
times = []
for _ in range(100):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

print(f"\nInference Speed (128x128 → 256x256):")
print(f"  Mean: {np.mean(times):.2f} ms")
print(f"  Median: {np.median(times):.2f} ms")
print(f"  FPS: {1000/np.mean(times):.0f}")

# Test with 256x256
dummy_256 = torch.randn(1, 1, 256, 256).to(device)
times_256 = []
for _ in range(50):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy_256)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times_256.append((time.perf_counter() - t0) * 1000)

print(f"\nInference Speed (256x256 → 512x512):")
print(f"  Mean: {np.mean(times_256):.2f} ms")
print(f"  FPS: {1000/np.mean(times_256):.0f}")


## 9. Push to GitHub

After training is complete and results look good:


In [ ]:
# # ── Push results to GitHub ────────────────────────────
# # Uncomment and run after everything is verified
#
# # Make sure outputs/.gitkeep exists
# !touch outputs/.gitkeep
#
# !git add -A
# !git status
# !git commit -m "Add trained weights and test outputs"
# !git push origin main


## 10. Fresh Environment Test (CRITICAL)

Before submission, verify everything works from scratch:


In [ ]:
# # ── Fresh env test (uncomment to run) ──────────────────
# import shutil
# TEST_DIR = '/content/fresh_test'
# if os.path.exists(TEST_DIR):
#     shutil.rmtree(TEST_DIR)
#
# !git clone https://github.com/<YOUR-USERNAME>/semicon-restore.git {TEST_DIR}
# %cd {TEST_DIR}
# !pip install -r requirements.txt -q
#
# # Test with a sample image
# !python evaluate.py \
#     --input_dir {DATA_DIR}/Test/Test_NoisyLR/In_Distribution \
#     --output_dir /content/fresh_test_out/ \
#     --weights weights/nafnet_tiny.pt
#
# print("\n✅ Fresh environment test PASSED!" if os.listdir('/content/fresh_test_out/') else "\n❌ FAILED!")
# %cd {REPO_DIR}


---
## ✅ Checklist Before Submission

- [ ] Training completed (~15k iters, PSNR ≥ 28 dB)
- [ ] `evaluate.py` runs clean with `--weights weights/nafnet_tiny.pt`
- [ ] Output images saved in `outputs/` folder
- [ ] `weights/nafnet_tiny.pt` committed to repo (< 25MB for GitHub)
- [ ] `README.md` has quickstart instructions
- [ ] Fresh clone + run test passes
- [ ] Slides PDF uploaded to portal
- [ ] GitHub link submitted

